In [4]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 將專案 root 加入 python path（讓 src/ 可以 import）
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)
print("✓ Imports ready")

Project root added: C:\Users\USER\Desktop\2026_japan
✓ Imports ready


In [5]:
from src.geo.grid_to_latlng import *

anchors = [
        {"x": 24, "y": 151, "lat": 43.06918333153887, "lng": 141.35147072116592},  # 札幌站 
        {"x": 24, "y": 148, "lat": 43.07940372979633, "lng": 141.34225589803765},  # 北海道大學
        {"x": 26, "y": 153, "lat": 43.05798589528942, "lng": 141.35402112326315},  # 狸小路商店街
        {"x": 52, "y": 81, "lat": 43.1982317547878, "lng": 140.99403634015297},  # 小樽站
        {"x": 50, "y": 41, "lat": 43.188064114901195, "lng": 140.79455411455163},  # 余市站
        {"x": 182, "y": 186, "lat": 43.85360951281324, "lng": 141.52348814480132},  # 増毛町文化センター
]

mapper = GridLatLngMapper(anchors)

In [ ]:
import pandas as pd
from src.parks_choose import *
from src.parks_create import *
from src.parks_routing import *

START_DATE = '2019-09-15'

south = 42.9
north = 43.9
west = 140.7
east = 141.6

df_pred = pd.read_parquet("../data/predictions/convlstm_df_pred.parquet")

df_parks, df_with_a, df_prob = create_poi_parking(df_pred,south,west,north,east,mapper,START_DATE) 


In [7]:
df_prob.head()

,d,t,park_id,demand,pressure,p_avail,park_x,park_y,capacity,is_city,urban_score
0,1,0,node/10095286778,0.026334,0.000239,0.880646,23.862832,147.013421,110,1,0.932311
1,1,0,node/10188151756,0.643720,0.025749,0.863599,36.004512,136.971182,25,1,0.392041
2,1,0,node/10673369415,0.026915,0.001035,0.880143,21.511308,151.410173,26,1,1.547945
3,1,0,node/10674905656,0.024629,0.000616,0.880409,21.142992,151.374945,40,1,0.911731
4,1,0,node/11134318196,0.160807,0.000696,0.880358,21.383037,157.735259,231,1,0.715780


In [8]:
import numpy as np
import pandas as pd

def prepare_sim_inputs(df_prob: pd.DataFrame):
    """
        把 df_prob 轉成：
        - steps: (d,t) 的時間步索引
        - parks_meta: 每個 park 的靜態資訊
        - demand_mat: shape = (n_steps, n_parks) 的 demand 矩陣
    """
    required = {"d", "t", "park_id", "demand", "capacity", "is_city"}
    missing = required - set(df_prob.columns)
    if missing:
        raise ValueError(f"df_prob 缺少欄位: {missing}")

    df = df_prob.copy()

    df["d"] = df["d"].astype(int)
    df["t"] = df["t"].astype(int)
    df["demand"] = df["demand"].astype(float)
    df["capacity"] = df["capacity"].astype(int)
    df["is_city"] = df["is_city"].astype(int)

    # 時間步
    steps = (df[["d", "t"]].drop_duplicates().sort_values(["d", "t"]).reset_index(drop=True))

    # 停車場 metadata（確保每個 park_id 唯一）
    meta_cols = ["park_id", "capacity", "is_city"]
    parks_meta = (df[meta_cols].drop_duplicates(subset=["park_id"]).set_index("park_id").sort_index())

    # demand 矩陣：index=(d,t), columns=park_id
    demand_pivot = (df.pivot_table(index=["d", "t"],columns="park_id",values="demand",aggfunc="mean",fill_value=0.0)
                      .reindex(pd.MultiIndex.from_frame(steps), fill_value=0.0))

    # 對齊欄位順序
    demand_pivot = demand_pivot.reindex(columns=parks_meta.index, fill_value=0.0)

    demand_mat = demand_pivot.to_numpy(dtype=float)

    return steps, parks_meta, demand_mat

# 停車模擬(demand = kappa * a * score, 單位:輛 / 30分鐘)
def simulate_parking_occupancy(
    df_prob: pd.DataFrame,
    *,
    slot_minutes: int = 30,
    arrival_scale: float = 2.0,             # 控制 demand 轉換為抵達車流的量級
    mean_stay_city_min: float = 90.0,       # 平均市區停車時間
    mean_stay_suburb_min: float = 120.0,    # 平均郊區停車時間
    init_occ_rate_city: float = 0.55,       # 市區初始已停車比例
    init_occ_rate_suburb: float = 0.35,     # 郊區初始已停車比例
    seed: int = 2025,
    return_matrix: bool = False
) -> pd.DataFrame:
    """
        單次模擬：輸出每個 (d,t,park_id) 的 occupancy 狀態與事件。
        
        模型假設：
        - arrivals ~ Poisson(λ), λ = arrival_scale * demand
        - departures ~ Binomial(occ, q), q = 1 - exp(-Δ / mean_stay)
        - 受容量限制 parked = min(arrivals, capacity - (occ - departures))
    """
    steps, parks_meta, demand_mat = prepare_sim_inputs(df_prob)

    rng = np.random.default_rng(seed)

    park_ids = parks_meta.index.to_numpy()
    cap = parks_meta["capacity"].to_numpy(dtype=int)
    is_city = parks_meta["is_city"].to_numpy(dtype=int)

    # 每個停車場的離場機率 q（依市區/郊區平均停車時間與時間步大小計算出）
    q_city = 1.0 - np.exp(-slot_minutes / mean_stay_city_min)
    q_sub  = 1.0 - np.exp(-slot_minutes / mean_stay_suburb_min)
    q = np.where(is_city == 1, q_city, q_sub).astype(float) #is_city 就用 q_city
    q = np.clip(q, 0.0, 1.0)

    # 初始化 occupancy
    init_rate = np.where(is_city == 1, init_occ_rate_city, init_occ_rate_suburb).astype(float)
    init_rate = np.clip(init_rate, 0.0, 0.95)  # 避免一開始就滿位
    occ = np.rint(cap * init_rate).astype(int)  # cap 四捨五入

    n_steps = demand_mat.shape[0]
    n_parks = demand_mat.shape[1]

    # 用矩陣來記錄結果
    occ_hist = np.zeros((n_steps, n_parks), dtype=int)   # 已停車數
    avail_hist = np.zeros((n_steps, n_parks), dtype=int) # 剩餘空位  
    arr_hist = np.zeros((n_steps, n_parks), dtype=int)   # 找位子車數
    dep_hist = np.zeros((n_steps, n_parks), dtype=int)   # 離開車數
    rej_hist = np.zeros((n_steps, n_parks), dtype=int)   # 滿位而無法停車數

    for i in range(n_steps):
        demand_row = demand_mat[i]  # 對每個時間步下，每個停車場的 demand

        # departures(把「每一台已停著的車在這個時間步內會不會離開」當成一個白努利試驗，而 n 台車的試驗總和就是二項式分布)
        dep = rng.binomial(n=occ, p=q)
        occ_after_dep = occ - dep

        # arrivals(抵達現象用卜瓦松分布描述)
        lam = np.maximum(0.0, arrival_scale * demand_row) # lambda : 每一個 time slot 的平均抵達車數
        arr = rng.poisson(lam=lam)

        # capacity constraint
        free = np.maximum(0, cap - occ_after_dep) # 剩餘空位
        parked = np.minimum(arr, free) # 來了 arr 台，但最多只能停 free 台
        rej = arr - parked

        # update occupancy
        occ = occ_after_dep + parked # 更新已停車數
        avail = cap - occ # 更新剩餘空位

        # record
        occ_hist[i] = occ
        avail_hist[i] = avail
        arr_hist[i] = arr
        dep_hist[i] = dep
        rej_hist[i] = rej

    if return_matrix:
        return steps, parks_meta, occ_hist, avail_hist  

    # 轉回 long format
    # 建 (d,t) 對應的索引
    dt_index = pd.MultiIndex.from_frame(steps, names=["d", "t"])

    df_out = []
    for j, pid in enumerate(park_ids):
        tmp = pd.DataFrame({
            "d": dt_index.get_level_values("d").to_numpy(),
            "t": dt_index.get_level_values("t").to_numpy(),
            "park_id": pid,
            "capacity": cap[j],
            "is_city": is_city[j],
            "occ": occ_hist[:, j],          # 該時間步已停車數 
            "available": avail_hist[:, j],  # 該時間步剩餘空位
            "arrivals": arr_hist[:, j],     # 該時間步來找位子車數
            "departures": dep_hist[:, j],   # 該時間步離開車數  
            "rejected": rej_hist[:, j],     # 該時間步因滿位而無法停車數
        })
        df_out.append(tmp)

    df_out = pd.concat(df_out, ignore_index=True)

    # optional meta
    if "urban_score" in parks_meta.columns:
        df_out = df_out.merge(parks_meta[["urban_score"]].reset_index(), on="park_id", how="left")
    if "park_x" in parks_meta.columns and "park_y" in parks_meta.columns:
        df_out = df_out.merge(parks_meta[["park_x","park_y"]].reset_index(), on="park_id", how="left")

    return df_out

# 蒙地卡羅模擬，用大量隨機試驗來預測不確定未來的平均結果
def monte_carlo_true_availability(
    df_prob: pd.DataFrame,
    *,
    n_runs: int = 200,
    seed: int = 2025,
) -> pd.DataFrame:
    """
        多次模擬，估計「真實可停車機率」：
        p_true_avail(d,t,park) = P(available > 0)
    """
    steps, parks_meta, _, avail0 = simulate_parking_occupancy(df_prob, seed = seed, return_matrix = True)
    n_steps, n_parks = avail0.shape

    avail_count = np.zeros((n_steps, n_parks), dtype=np.int32) # 第 i 個時間步、第 j 個停車場，有位子的次數
    occ_sum = np.zeros((n_steps, n_parks), dtype=np.float64) # 累計已停車數之和，最後除以 runs 得平均

    for r in range(n_runs):
        _, _, occ_hist, avail_hist = simulate_parking_occupancy(df_prob, seed = seed + r, return_matrix = True)
        avail_count += (avail_hist > 0).astype(np.int32)
        occ_sum += occ_hist

    p_true = avail_count / n_runs
    occ_mean = occ_sum / n_runs

    dt_index = pd.MultiIndex.from_frame(steps, names=["d","t"])
    park_ids = parks_meta.index.to_numpy()

    out = []
    for j, pid in enumerate(park_ids):
        out.append(pd.DataFrame({
            "d": dt_index.get_level_values("d").to_numpy(),
            "t": dt_index.get_level_values("t").to_numpy(),
            "park_id": pid,
            "p_true_avail": p_true[:, j],
            "occ_mean": occ_mean[:, j],
            "capacity": parks_meta.loc[pid, "capacity"],
            "is_city": parks_meta.loc[pid, "is_city"],
        }))
    df_true = pd.concat(out, ignore_index=True)

    return df_true

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) 用你的 Monte Carlo 產生「真實可停機率」
df_true = monte_carlo_true_availability(df_prob, n_runs=200, seed=2025)

# 2) 合併模型輸出 p_avail（sigmoid）與真實 p_true_avail
df_eval = df_prob.merge(
    df_true[["d","t","park_id","p_true_avail","occ_mean"]],
    on=["d","t","park_id"],
    how="left"
)

# 3) 基本檢查
assert df_eval["p_true_avail"].notna().all(), "有合併不到的 (d,t,park_id)，先檢查 df_prob/df_true keys 是否一致"

In [11]:
df_eval.head()

,d,t,park_id,demand,pressure,p_avail,park_x,park_y,capacity,is_city,urban_score,p_true_avail,occ_mean
0,1,0,node/10095286778,0.026334,0.000239,0.880646,23.862832,147.013421,110,1,0.932311,1.0,43.470
1,1,0,node/10188151756,0.643720,0.025749,0.863599,36.004512,136.971182,25,1,0.392041,1.0,11.380
2,1,0,node/10673369415,0.026915,0.001035,0.880143,21.511308,151.410173,26,1,1.547945,1.0,10.060
3,1,0,node/10674905656,0.024629,0.000616,0.880409,21.142992,151.374945,40,1,0.911731,1.0,15.985
4,1,0,node/11134318196,0.160807,0.000696,0.880358,21.383037,157.735259,231,1,0.715780,1.0,91.670


In [13]:
df_eval['p_true_avail'].describe()

count    1.392739e+07
mean     9.978797e-01
std      3.120254e-02
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: p_true_avail, dtype: float64